# 58. `CoherentAmplitudeModel` built by hand

**Objectives:**

- Build a coherent sum `A(x) = sum_i c_i F_i(x)` directly from `AmplitudeComponent` and
  `CoherentAmplitudeModel`, bypassing `DecayModel`/`Resonance` entirely.
- Use `ConstantAmplitude` for a trivial dynamical shape and a small hand-written function for a
  second one.
- Evaluate `.amplitude(...)`/`.intensity(...)` directly on a plain dict of Dalitz points, and
  contrast this minimal path with the higher-level `Resonance`/`DecayModel` machinery used
  elsewhere in this course.

Run the cells in order in a fresh kernel. Masses are in GeV, invariants in GeV^2.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede any numerical work: amplitudes use complex128.

import jax.numpy as jnp

from dalitzplotfitter import (
    AmplitudeComponent, CoherentAmplitudeModel, ConstantAmplitude,
    DecayChannel, DecayModel, NonResonant, RealImag, Resonance, generate_toy,
)

## 1. The minimal manual construction path

`AmplitudeComponent(name, function, coefficient)` pairs a bare dynamical function
`function(data, parameters) -> Array` with a coefficient. `coefficient_value` (in
`amplitude/components.py`) resolves the coefficient through its own `.value(values)` method if
it has one (e.g. `RealImag`), or otherwise uses it directly as a fixed scalar -- a plain
Python/JAX complex number works just as well as a `RealImag` object when nothing needs to be
floated. `ConstantAmplitude()` is the simplest possible `function`: a non-resonant, momentum
-independent unit shape, broadcast to one value per event.

In [2]:
nr_component = AmplitudeComponent(
    name="NR",
    function=ConstantAmplitude(),
    coefficient=0.4 - 0.2j,  # a plain fixed complex coefficient, no RealImag needed here
)

def hand_rolled_bw(data, parameters=None):
    """A bare relativistic-Breit-Wigner-shaped function over s12, with no barrier
    factors, angular terms or normalization -- everything `Resonance` adds on top."""
    del parameters
    s12 = data["s12"]
    m0, w0 = 0.7753, 0.1491
    return 1.0 / (m0**2 - s12 - 1j * m0 * w0)

rho_component = AmplitudeComponent(
    name="rho", function=hand_rolled_bw, coefficient=1.0 + 0.3j,
)

manual_model = CoherentAmplitudeModel(components=(nr_component, rho_component))
print("Components:", [c.name for c in manual_model.components])

Components: ['NR', 'rho']


## 2. Evaluate directly on a small array of Dalitz points

`CoherentAmplitudeModel.amplitude`/`.intensity` take a plain `data` mapping (`{"s12": ..., ...}`,
exactly the shape `PhaseSpaceSample.as_dict()` returns) -- no `DecayChannel`, no barrier factors,
no `PreparedAmplitudeCache`, no normalization matrix. A handful of physical points is enough; we
borrow them from a throwaway `DecayModel`'s toy generator purely as a convenient source of
points that are guaranteed to lie in the physical `D+ -> pi- pi+ pi+` region.

In [3]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
throwaway_model = DecayModel(
    channel,
    components=[Resonance("rho", (0, 1), RealImag(1.0, 0.0), mass=0.7753, width=0.1491, spin=1)],
    normalization_method="square-dalitz", normalization_resolution=20,
)
points = generate_toy(
    throwaway_model, 6, seed=58, method="inverse-transform",
    inverse_resolution=200, include_momenta=False,
)
data = points.as_dict()
print("data keys:", list(data.keys()), " n events:", data["s12"].shape[0])

amplitude = manual_model.amplitude(data)
intensity = manual_model.intensity(data)
for i in range(data["s12"].shape[0]):
    print(f"s12={float(data['s12'][i]):.4f}  A={complex(amplitude[i]):.4f}  |A|^2={float(intensity[i]):.4f}")

data keys: ['s12', 's13', 's23']  n events: 6
s12=0.4690  A=3.5618+4.8367j  |A|^2=36.0800
s12=0.7775  A=-4.3447+1.2081j  |A|^2=20.3358
s12=0.5982  A=-1.9797+8.5096j  |A|^2=76.3327
s12=0.3732  A=3.3591+2.6178j  |A|^2=18.1364
s12=2.2004  A=-0.2355-0.3416j  |A|^2=0.1722
s12=0.4388  A=3.6143+3.9384j  |A|^2=28.5741


## 3. Contrast with the `Resonance`/`DecayModel` path

The manual sum above evaluates one bare relativistic-Breit-Wigner shape with *no* barrier
factors, no angular/spin factor, no per-pair mass ordering logic, and no normalization -- an
uncached, un-integrated coherent amplitude used purely for the numerator `|A|^2`.
`Resonance`/`DecayModel` build on exactly this same `CoherentAmplitudeModel` engine
underneath, but add: `lineshape` + `angular` composition (Blatt-Weisskopf barriers, spin
factors), per-component unit-integral rescaling (`normalize_component`), the Hermitian
normalization matrix `M_ij = integral(conj(F_i) F_j) dPhi` needed to turn `|A|^2` into a proper
likelihood, and `PreparedAmplitudeCache`'s fixed/floating split for fast re-fits. Compare the
manual `rho` shape used above against `Resonance`'s own lineshape evaluated on the same points,
to see exactly what the low-level path is missing.

In [4]:
from dalitzplotfitter import RelativisticBreitWigner, ResonanceContext

mpi = 0.13957
context = ResonanceContext(
    parent_mass=1.869, daughter_masses=(mpi, mpi), bachelor_mass=mpi,
    spin=1, pole_mass=0.7753, pole_width=0.1491,
)
m12 = jnp.sqrt(data["s12"])
resonance_shape = RelativisticBreitWigner()(m12, context)
bare_shape = 1.0 / (0.7753**2 - data["s12"] - 1j * 0.7753 * 0.1491)

print(f"{'s12':>8s} {'bare 1/(m0^2-s-i m0 w0)':>26s} {'Resonance lineshape (with barrier/spin)':>42s}")
for i in range(data["s12"].shape[0]):
    print(f"{float(data['s12'][i]):8.4f} {complex(bare_shape[i]):26.4f} {complex(resonance_shape[i]):42.4f}")
print()
print("The two differ because RelativisticBreitWigner (as used inside Resonance) folds in the "
      "P-wave angular/spin factor and Blatt-Weisskopf barrier terms that the bare manual "
      "function above deliberately omits.")

     s12    bare 1/(m0^2-s-i m0 w0)    Resonance lineshape (with barrier/spin)
  0.4690             4.2870+3.7506j                             5.1798+3.5173j
  0.7775            -3.9654+2.5977j                            -3.3644+2.7834j
  0.5982             0.2140+8.6454j                             0.2159+8.6852j
  0.3732             3.4903+1.7707j                             4.0195+1.2181j
  2.2004            -0.6220+0.0450j                            -0.6052+0.1102j
  0.4388             4.0879+2.9120j                             4.8748+2.5052j

The two differ because RelativisticBreitWigner (as used inside Resonance) folds in the P-wave angular/spin factor and Blatt-Weisskopf barrier terms that the bare manual function above deliberately omits.


## Continue learning

See `dalitzplotfitter/amplitude/components.py` for `AmplitudeComponent`,
`CoherentAmplitudeModel` and `ConstantAmplitude`'s docstrings, and
[`docs/dynamics_structure.md`](../../docs/dynamics_structure.md) for how `Resonance` composes
`lineshape`/`angular` on top of this same low-level engine.

Return to [the course guide](TUTORIALS.md).